# B2-024-gpu-scientific-ml-capstone — Practice p22 — Solution

**Type:** integrative · **Difficulty:** core · **Concepts:** open-ended-experiment-design

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

import time

TOTAL_BUDGET = 2000
MAX_WIDTH = 64

## Solution

**Design.** Three configurations of the same `8 → 64 → 64 → 3` MLP (width 64, the maximum allowed), differing only in the input representation:

* `raw` — standardized raw magnitudes;
* `log` — standardized log magnitudes (the reference);
* `log+aug` — `log` plus Gaussian noise of standard deviation $0.03$ added to the log magnitudes at every step (fresh draws from the global generator), before standardization.

Each pair of neighbours differs in one factor (`raw`→`log`: the transform; `log`→`log+aug`: the augmentation). All runs share seeds 0 and 1, full-batch Adam at $3\times10^{-3}$, 300 steps, training statistics only, and the validation metric.
The hypothesis and decision rule are written down as strings before the first run, and the decision is applied by code to the log.

In [ ]:
import math

train_x, train_y = capstone_data.train_rows("inverse")
val_x, val_y = capstone_data.val_rows("inverse")
SEAM = []


class InverseMLP:
    def __init__(self, config, seed, steps, budget):
        self.config, self.seed, self.steps, self.budget = config, seed, steps, budget

    def _feats(self, m):
        return torch.log(m) if self.config["use_log"] else m

    def fit(self, train_x, train_y):
        SEAM.extend(capstone_data.split_of("inverse", row) for row in train_x)
        f = self._feats(train_x)
        self.mu, self.sd = f.mean(dim=0), f.std(dim=0)            # training statistics only
        torch.manual_seed(self.seed)
        width = self.config["width"]
        assert width <= MAX_WIDTH
        self.net = nn.Sequential(nn.Linear(8, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU(), nn.Linear(width, 3))
        opt = self.budget.wrap(torch.optim.Adam(self.net.parameters(), lr=3e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0))
        self.net.train()
        for _ in range(self.steps):
            inp = f + self.config["aug_std"] * torch.randn_like(f) if self.config["aug_std"] > 0 else f
            opt.zero_grad(set_to_none=True)
            loss = F.mse_loss(self.net((inp - self.mu) / self.sd), train_y)
            loss.backward()
            opt.step()
        self.net.eval()
        return self

    def predict(self, x):
        with torch.no_grad():
            return self.net((self._feats(x) - self.mu) / self.sd)


CONFIGS = {
    "raw": {"use_log": False, "aug_std": 0.0, "width": 64},
    "log": {"use_log": True, "aug_std": 0.0, "width": 64},
    "log+aug": {"use_log": True, "aug_std": 0.03, "width": 64},
}
CAMPAIGN_SEEDS = (0, 1)
STEPS = 300
LOG = []

HYPOTHESIS = ("Change: add Gaussian input-noise augmentation (std 0.03 on the log magnitudes, fresh noise every step) to the "
              "8-64-64-3 log-magnitude MLP. Metric and direction: mean validation source-position error decreases. "
              "Minimum effect: at least 5% relative to the 'log' reference. Protocol: training rows only, statistics from "
              "training rows, full-batch Adam lr 3e-3, 300 steps per run, shared seeds 0 and 1, one StepBudget(2000); "
              "the 'raw' configuration is the ablation of the log transform.")
DECISION_RULE = ("Reference is 'log'. A configuration replaces it only if its mean validation error over seeds is at least 5% "
                 "below the reference's mean AND it beats the reference on every shared seed; among such configurations choose "
                 "the lowest mean (ties -> the simpler configuration, in the order raw, log, log+aug). If none qualifies the "
                 "improvement is 'not demonstrated' and 'log' is kept.")
LOG_LEN_AT_DEFINITION = len(LOG)
print("HYPOTHESIS defined; len(LOG) =", LOG_LEN_AT_DEFINITION)

In [ ]:
budget = capstone_data.StepBudget(TOTAL_BUDGET)
# Budget arithmetic: 3 configurations x 2 seeds x 300 steps = 1800 steps <= 2000 (200 steps of slack, unused).
assert len(CONFIGS) * len(CAMPAIGN_SEEDS) * STEPS <= TOTAL_BUDGET
MODELS = {}
for name, config in CONFIGS.items():
    for seed in CAMPAIGN_SEEDS:
        start = time.perf_counter()
        used_before = budget.used
        model = InverseMLP(config, seed, STEPS, budget).fit(train_x, train_y)
        val_error = capstone_data.position_error_per_example(model.predict(val_x), val_y).mean().item()
        MODELS[name, seed] = model
        LOG.append({"run": len(LOG) + 1, "config": name, "seed": seed, "steps": budget.used - used_before,
                    "val_error": val_error, "seconds": time.perf_counter() - start,
                    "note": f"use_log={config['use_log']}, aug_std={config['aug_std']}, width={config['width']}"})
for rec in LOG:
    print(f"run {rec['run']}: {rec['config']:8s} seed {rec['seed']} steps {rec['steps']} val {rec['val_error']:.4f} ({rec['seconds']:.2f}s) {rec['note']}")


def decide(log):
    order = ["raw", "log", "log+aug"]
    by = {}
    for rec in log:
        by.setdefault(rec["config"], {})[rec["seed"]] = rec["val_error"]
    mean = {c: sum(v.values()) / len(v) for c, v in by.items()}
    ref = "log"
    qualified = [c for c in by if c != ref
                 and mean[c] <= 0.95 * mean[ref]
                 and all(by[c][s] < by[ref][s] for s in by[ref])]
    if not qualified:
        return ref
    return min(qualified, key=lambda c: (mean[c], order.index(c)))


chosen = decide(LOG)
means = {c: sum(r["val_error"] for r in LOG if r["config"] == c) / len(CAMPAIGN_SEEDS) for c in CONFIGS}
print("means:", {c: round(m, 4) for c, m in means.items()}, "| decision:", chosen, "| budget used:", budget.used)

**Locked test score of the chosen configuration's seed-0 model (one call, no retraining).**

In [ ]:
baseline = capstone_data.baseline_score("inverse")
scored_config = chosen
scored_model = MODELS[scored_config, 0]
assert capstone_data.test_call_count("inverse") == 0
test = capstone_data.final_test_score(scored_model.predict, n_boot=1000, alpha=0.05, task="inverse")
assert capstone_data.test_call_count("inverse") == 1
print(f"{scored_config} seed 0: test position error {test.score:.4f}, 95% CI [{test.ci_low:.4f}, {test.ci_high:.4f}] (baseline {baseline.score:.4f})")

**Campaign summary.**

In [ ]:
per = {c: [r["val_error"] for r in LOG if r["config"] == c] for c in CONFIGS}
fmt = lambda c: "/".join(f"{v:.4f}" for v in per[c])
rel = lambda c: 100 * (means[c] / means["log"] - 1)
log_vs_raw = all(a < b for a, b in zip(per["log"], per["raw"]))
aug_demo = decide([r for r in LOG if r["config"] in ("log", "log+aug")]) == "log+aug"
summary = (
    f"We tested whether log features and, on top of them, input-noise augmentation (std 0.03) lower the mean validation position error of a width-64 MLP, with three configurations, shared seeds 0 and 1, and 300 full-batch steps per run (1,800 of 2,000 budgeted steps). "
    f"The log shows per-seed validation errors (seed 0/seed 1) of {fmt('raw')} for raw, {fmt('log')} for log, and {fmt('log+aug')} for log+aug. "
    f"Log features {'beat' if log_vs_raw else 'did not consistently beat'} raw magnitudes on both seeds ({rel('raw'):+.0f}% for raw relative to log), "
    f"while augmentation changed the mean by {rel('log+aug'):+.1f}%, which {'meets' if aug_demo else 'does not meet'} the pre-registered bar (a mean reduction of at least 5% and a win on every shared seed), so its benefit is {'demonstrated' if aug_demo else 'not demonstrated'}. "
    f"Applying the decision rule to the log selects '{chosen}', whose seed-0 model scored {test.score:.4f} on the locked test set (95% CI [{test.ci_low:.4f}, {test.ci_high:.4f}]) against the grid-Tikhonov baseline's {baseline.score:.4f}. "
    f"With a further 2,000 steps I would run log versus log+aug with four more shared seeds at 250 steps each (2 x 4 x 250 = 2,000) to estimate the augmentation effect's sign and size with a paired comparison, since two seeds cannot separate a few-percent effect from seed noise."
)
print(summary)

### Answer check

In [ ]:
KEYS = {"run", "config", "seed", "steps", "val_error", "seconds", "note"}
assert budget.used <= TOTAL_BUDGET
assert len({rec["steps"] for rec in LOG}) == 1 and LOG[0]["steps"] == STEPS
assert len(LOG) == len(CONFIGS) * len(CAMPAIGN_SEEDS) == 6
assert all(set(rec) == KEYS for rec in LOG)
assert len(CONFIGS) >= 3 and len(CAMPAIGN_SEEDS) >= 2
assert isinstance(HYPOTHESIS, str) and HYPOTHESIS and isinstance(DECISION_RULE, str) and DECISION_RULE
assert LOG_LEN_AT_DEFINITION == 0
assert decide(LOG) == scored_config
assert set(SEAM) == {"train"} and len(SEAM) == 6 * 800
assert capstone_data.test_call_count("inverse") == 1